In [1]:
# Imports
import re
import warnings
from pathlib import Path

import pandas as pd

from raptcr.io.pipeline import ProcessingPipeline
from raptcr.io.mappers import RegexMapper

from utils.conv_clustering import parse_mixcr, load_genus_results

# Specify settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

2025-09-29 14:36:15 - RepertoireReader - INFO - Logging initialized


In [2]:
base_dir = Path("..").resolve()

# Discovery dataset directories
disc_data_dir = base_dir / "1_raw_data" / "0_discovery_data"
disc_raw_tcr_dir = disc_data_dir / "0_discovery_TCR"
disc_conv_dir = base_dir / "2_processed_data" / "3_discovery_convergence"
disc_conv_dir.mkdir(exist_ok=True)

# Brand et al. IBD dataset
ibd_data_dir = base_dir / "1_raw_data" / "1_Brand_data"
ibd_processed_dir = base_dir / "2_processed_data" / "1_Brand_data"
ibd_conv_dir = base_dir / "2_processed_data" / "4_Brand_convergence"
ibd_conv_dir.mkdir(exist_ok=True)

# Pu et al. CRC dataset
crc_data_dir = base_dir / "1_raw_data" / "2_Pu_data"
crc_processed_dir = base_dir / "2_processed_data" / "2_Pu_data"
crc_conv_dir = base_dir / "2_processed_data" / "5_Pu_convergence"
crc_conv_dir.mkdir(exist_ok=True)

# 1. Discovery dataset preparation

### 1A. Get discovery TCR data

In [3]:
# Input files
overlap_file = disc_data_dir / "Overlap_id_list.csv"
microbiome_file = disc_data_dir / "Microbiome_full_names_taxa.csv"

# Output files
alpha_out = disc_conv_dir / "TRA_dataframe.csv"
beta_out = disc_conv_dir / "TRB_dataframe.csv"
genus_mtx = disc_conv_dir / "genus_count_matrix_th1.csv"

# Process TCR repertoires with TRIASSIC
patient_id_mapper = RegexMapper(pattern=r"(HH\d+|CO\d+|PT\d+)")
repertoire_id_mapper = RegexMapper(pattern=r"[-_]([Covid|ERC].*[_-]CD[48])")

pipe = ProcessingPipeline(
    patient_id_mapping=patient_id_mapper,
    repertoire_id_mapping=repertoire_id_mapper,
)

# Use the raw TCR data
disc_data = pipe.process_dataset(disc_raw_tcr_dir)

# Add chain label (TRA/TRB) and fix IDs
disc_data["chain"] = disc_data["v_gene"].str[:3]
disc_data["patient_id"] = disc_data["patient_id"].replace(
    {"HH0781": "HH078.1", "HH0782": "HH078.2"}
)
disc_data["repertoire_id"] = disc_data["repertoire_id"].replace(
    {"ERC-CO182-CD4": "ERC_CO182_CD4", "ERC-CO182-CD8": "ERC_CO182_CD8"}
)

# Filter to patients with overlapping microbiome data
overlapping = pd.read_csv(overlap_file)
disc_data = disc_data[disc_data["patient_id"].isin(overlapping["patient_id"])]

# Split into alpha and beta repertoires
data_alpha = disc_data.query("chain == 'TRA'").reset_index(drop=True)
data_beta = disc_data.query("chain == 'TRB'").reset_index(drop=True)

# Format the V- and J-genes
for df in (data_alpha, data_beta):
    df["v_call"] = df["v_gene"] + "*01"
    df["j_call"] = df["j_gene"] + "*01"

# Save processed repertoires
data_alpha.to_csv(alpha_out, index=False)
data_beta.to_csv(beta_out, index=False)

2025-09-29 14:36:16 - RepertoireReader - INFO - Read 145930 sequences with reader:mixcr
2025-09-29 14:36:16 - RepertoireReader - INFO - Filtered 12245 (8.39%) rows with missing values
2025-09-29 14:36:16 - RepertoireReader - INFO - Filtered 514 (0.38%) rows with invalid V/J genes
2025-09-29 14:36:16 - RepertoireReader - INFO - Filtered 15951 (11.98%) rows with invalid junction_aa's
2025-09-29 14:36:16 - RepertoireReader - INFO - Filtered 1593 (1.36%) rows with non-productive V/J genes
2025-09-29 14:36:17 - RepertoireReader - INFO - Trimmed 115627 (1.00) junction_nt to CDR3 sequence
2025-09-29 14:36:17 - RepertoireReader - INFO - Grouped 13985 (12.09%) clonotypes based on (['junction', 'v_gene', 'junction_aa', 'j_gene'])
2025-09-29 14:36:17 - RepertoireReader - INFO - Read 33565 sequences with reader:mixcr
2025-09-29 14:36:17 - RepertoireReader - INFO - Filtered 2460 (7.33%) rows with missing values
2025-09-29 14:36:17 - RepertoireReader - INFO - Filtered 791 (2.54%) rows with invalid V

### 1B. Get genus level microbiom count matrix

In [4]:
# Get all patient IDs with paired TCR and microbiome data
patient_list = overlapping["patient_id"]

# Read in the microbiome count matrix
full_micro = pd.read_csv(microbiome_file)

# Select those samples that also have TCR data
full_micro_overlap = full_micro[full_micro["subject"].isin(patient_list)]

# Extract genus level and aggregate abundance at the genus level
full_micro_overlap["genus"] = full_micro_overlap["taxon_name"].str.split(" ").str[0]
matrix_genus = (
    full_micro_overlap.groupby(["genus", "subject"])["rel_abundance"]
    .sum()
    .reset_index()
)
matrix_genus = pd.pivot_table(
    matrix_genus, values="rel_abundance", index="genus", columns="subject", fill_value=0
)

# Keep only genera found in more than one subject
matrix_genus = matrix_genus[(matrix_genus != 0).sum(axis=1) > 1]

# Save genus abundance matrix
matrix_genus.T.to_csv(genus_mtx)

# 2. Brand et al. data preparation

In [5]:
# Load the Brand et al. TCR data
ibd_data = pd.read_csv(ibd_processed_dir / "0_parsed_full_IBD_data.csv")

# Adjust metadata to fill empty values
ibd_data["Concordancy"] = ibd_data["Concordancy"].fillna("Healthy")
ibd_data.loc[ibd_data["Concordancy"] == "Healthy", "Diagnosis"] = "Healthy"

# Select the correct columns and samples
ibd_data = ibd_data[
    [
        "junction",
        "v_call",
        "junction_aa",
        "j_call",
        "patient_id",
        "Diagnosis",
        "chain_corrected",
    ]
]

twin_samples = [
    "TWIN0001",
    "TWIN0002",
    "TWIN0011",
    "TWIN0012",
    "TWIN0015",
    "TWIN0016",
    "TWIN0041",
    "TWIN0042",
    "TWIN0045",
    "TWIN0046",
    "TWIN0081",
    "TWIN0082",
    "TWIN0099",
    "TWIN0100",
    "TWIN0101",
    "TWIN0102",
]

# Split into alpha and beta dataframes
ibd_data_alpha = ibd_data[
    (ibd_data["chain_corrected"] == "TRA") & (ibd_data["patient_id"].isin(twin_samples))
]
ibd_data_beta = ibd_data[
    (ibd_data["chain_corrected"] == "TRB") & (ibd_data["patient_id"].isin(twin_samples))
]

# Save data
ibd_data_alpha.to_csv(ibd_conv_dir / "TRA_dataframe.csv", index=False)
ibd_data_beta.to_csv(ibd_conv_dir / "TRB_dataframe.csv", index=False)

In [6]:
# Load microbiome counts
ibd_micro = pd.read_excel(ibd_data_dir / "2_microbiome_species.xlsx", index_col=0)

# Extract microbiome species level
ibd_micro.columns = ibd_micro.columns.str.extract(r"\.s__(.+)$")[0]

# Extract the genus name
ibd_micro.columns = ibd_micro.columns.str.extract(r"^([^_]+)_")[0]

# Aggregate at the genus level
ibd_micro = ibd_micro.T.groupby(level=0).sum().T

# Keep only genera present in at least 2 individuals
ibd_micro = ibd_micro.loc[:, (ibd_micro != 0).sum(axis=0) > 1]

ibd_micro.to_csv(ibd_conv_dir / "genus_count_matrix_th1.csv")

# 3. Pu et al. data preparation

In [7]:
crc_data = pd.read_csv(crc_processed_dir / "0_Janssen_TCR.csv")

# Drop duplicate entries based on the patient and their clone ID, different cells will be kept
crc_data = crc_data.drop_duplicates(subset=["sample_tissue", "clone_id_aa_identity"])

# Split the single cell dataframe into alpha and beta sequences
crc_alpha = (
    crc_data[
        [
            "patient_id",
            "tissue",
            "sample_tissue",
            "MSS_type",
            "iCMS_type",
            "majorSTF",
            "cell_type",
            "IR_VJ_1_junction",
            "IR_VJ_1_junction_aa",
            "IR_VJ_1_v_call",
            "IR_VJ_1_j_call",
            "clone_id_aa_identity",
            "full_tcr",
        ]
    ]
    .rename(
        columns={
            "IR_VJ_1_junction": "junction",
            "IR_VJ_1_junction_aa": "junction_aa",
            "IR_VJ_1_v_call": "v_call",
            "IR_VJ_1_j_call": "j_call",
            "sample_tissue": "repertoire_id",
        }
    )
    .dropna(subset=["junction_aa"])
)
crc_beta = (
    crc_data[
        [
            "patient_id",
            "tissue",
            "sample_tissue",
            "MSS_type",
            "iCMS_type",
            "majorSTF",
            "cell_type",
            "IR_VDJ_1_junction",
            "IR_VDJ_1_junction_aa",
            "IR_VDJ_1_v_call",
            "IR_VDJ_1_j_call",
            "clone_id_aa_identity",
            "full_tcr",
        ]
    ]
    .rename(
        columns={
            "IR_VDJ_1_junction": "junction",
            "IR_VDJ_1_junction_aa": "junction_aa",
            "IR_VDJ_1_v_call": "v_call",
            "IR_VDJ_1_j_call": "j_call",
            "sample_tissue": "repertoire_id",
        }
    )
    .dropna(subset=["junction_aa"])
)


crc_alpha = parse_mixcr(crc_alpha, rename=False)
crc_alpha = crc_alpha.dropna().reset_index(drop=True)
crc_beta = parse_mixcr(crc_beta, rename=False)
crc_beta = crc_beta.dropna().reset_index(drop=True)

crc_alpha.to_csv(crc_conv_dir / "TRA_dataframe.csv")
crc_beta.to_csv(crc_conv_dir / "TRB_dataframe.csv")

In [8]:
crc_micro = pd.read_csv(
    crc_processed_dir / "1_Micro_count_matrix_full.csv", index_col=0
)

# Extract the genera as column names
crc_micro.columns = crc_micro.columns.str.extract(r"^(.+?) ")[0]

# Sum the counts per genus, then only keep the genera that are present in at least 2 different patients
crc_micro = crc_micro.T.groupby(level=0).sum().T
crc_micro = crc_micro.loc[:, (crc_micro != 0).sum(axis=0) > 1]

crc_micro.to_csv(crc_conv_dir / "genus_count_matrix_th1.csv")

# 4. Get shared genera and calculate convergence per genus

### 4A. Get the microbiome matrices of all three datasets

Compare the genus-level count matrices and extract genera present in all 3 of the datasets

In [9]:
# Load genus count matrices
disc_matrix = pd.read_csv(disc_conv_dir / "genus_count_matrix_th1.csv", index_col=0)
ibd_matrix = pd.read_csv(ibd_conv_dir / "genus_count_matrix_th1.csv", index_col=0)
crc_matrix = pd.read_csv(crc_conv_dir / "genus_count_matrix_th1.csv", index_col=0)

# Identify overlapping genera across datasets
disc_list, ibd_list, crc_list = map(
    set, [disc_matrix.columns, ibd_matrix.columns, crc_matrix.columns]
)
overlap_list = list(disc_list & ibd_list & crc_list)

print(
    f"ERC data: {len(disc_list)} unique genera\n"
    f"IBD data: {len(ibd_list)} unique genera\n"
    f"Janssen data: {len(crc_list)} unique genera\n"
    f"Total overlapping genera: {len(overlap_list)}"
)

# Keep only overlapping genera
for matrix, conv_dir in zip(
    [disc_matrix, ibd_matrix, crc_matrix], [disc_conv_dir, ibd_conv_dir, crc_conv_dir]
):
    matched_matrix = matrix.loc[:, matrix.columns.isin(overlap_list)]
    matched_matrix.to_csv(conv_dir / "genus_count_matrix_matched.csv")

ERC data: 300 unique genera
IBD data: 52 unique genera
Janssen data: 321 unique genera
Total overlapping genera: 36


### 4B. Calculate convergence for each of the 36 shared genera separately

Use these shared genera in the script 3A_genus_convergence.py to calculate convergence values for each genus in each dataset.

Takes +- 10 hours to run the full analysis

(This does not include Peptoniphilus since this is not present in the twins dataset)

(Also we removed Bacteroides from the list (later) since it was present in all individuals, leading to no negative group in the analysis)

### 4C. Load the convergence results for each genus and each dataset

This includes the TCR results from the analysis of all 36 genera and the TCRs for specifically Prevotella bivia and Peptoniphilus lacrimalis ("Selected" data)

In [10]:
# Load discovery, crc, and ibd genus convergence results
disc_results_dir = (
    base_dir / "2_processed_data" / "3_discovery_convergence" / "genus_results"
)
ibd_results_dir = (
    base_dir / "2_processed_data" / "4_Brand_convergence" / "genus_results"
)
crc_results_dir = base_dir / "2_processed_data" / "5_Pu_convergence" / "genus_results"


combined_disc = load_genus_results(disc_results_dir, file_selection="full_genera")
combined_disc["dataset"] = "Discovery"
combined_disc_selected = load_genus_results(
    disc_results_dir, file_selection="selected_genera"
)
combined_disc_selected["dataset"] = "Discovery"

combined_ibd = load_genus_results(ibd_results_dir, file_selection="full_genera")
combined_ibd["dataset"] = "Brand"
combined_ibd_selected = load_genus_results(
    ibd_results_dir, file_selection="selected_genera"
)
combined_ibd_selected["dataset"] = "Brand"

combined_crc = load_genus_results(crc_results_dir, file_selection="full_genera")
combined_crc["dataset"] = "Pu"
combined_crc_selected = load_genus_results(
    crc_results_dir, file_selection="selected_genera"
)
combined_crc_selected["dataset"] = "Pu"

# Combine all datasets
combined = pd.concat([combined_disc, combined_ibd, combined_crc])
combined_selected = pd.concat(
    [combined_disc_selected, combined_ibd_selected, combined_crc_selected]
)

combined_selected["genus"] = combined_selected["genus"].str.split("_").str[0]

final_conv = pd.concat([combined, combined_selected], ignore_index=True)

final_conv["repertoire_id"] = final_conv["repertoire_id"].replace(
    {"ERC-CO182-CD4": "ERC_CO182_CD4", "ERC-CO182-CD8": "ERC_CO182_CD8"}
)

# Reorder columns
final_conv = final_conv[
    [
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "patient_id",
        "repertoire_id",
        "dataset",
        "Diagnosis",
        "tissue",
        "MSS_type",
        "iCMS_type",
        "majorSTF",
        "cell_type",
        "clone_id_aa_identity",
        "full_tcr",
        "match_true",
        "match_false",
        "background_true",
        "background_false",
        "statistic",
        "pvalue",
        "convergence",
        "genus",
    ]
]

final_conv = final_conv.drop_duplicates()

# Save combined results
output_dir = base_dir / "2_processed_data/6_combined_convergence"
output_dir.mkdir(exist_ok=True)
final_conv.to_csv(output_dir / "0_combined_final_sign_interactions.csv", index=False)

Desulfovibrio_alpha_results.csv 17983
Faecalibacterium_beta_results.csv 0
Coprobacter_beta_results.csv 2808
Coprococcus_alpha_results.csv 10163
Erysipelotrichaceae_beta_results.csv 83654
Faecalibacterium_alpha_results.csv 725
Odoribacter_alpha_results.csv 6241
Prevotella_beta_results.csv 2867
Parasutterella_beta_results.csv 3156
Ruminococcus_beta_results.csv 1822
Dorea_alpha_results.csv 1661
Phascolarctobacterium_beta_results.csv 5971
Ruminococcus_alpha_results.csv 6230
Collinsella_beta_results.csv 3122
Paraprevotella_alpha_results.csv 26474
Coprococcus_beta_results.csv 11128
Bifidobacterium_alpha_results.csv 7512
Sutterella_alpha_results.csv 10331
Alistipes_beta_results.csv 59
Parasutterella_alpha_results.csv 6705
Pseudoflavonifractor_beta_results.csv 0
Eubacterium_alpha_results.csv 40589
Oscillibacter_beta_results.csv 0
Prevotella_alpha_results.csv 6238
Clostridium_beta_results.csv 5121
Odoribacter_beta_results.csv 349
Porphyromonas_alpha_results.csv 41599
Eubacterium_beta_results.cs

# 5. Cluster and plot for each genus separately

In [ ]:
## Run this in a virtual environment with ClusTCR installed ##
from pathlib import Path
import pandas as pd
import warnings

from utils.conv_clustering import get_clusters, analyze_clusters

# Specify settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [2]:
# Load the convergent TCR data
base_dir = Path("..").resolve()

final_conv = pd.read_csv(
    base_dir
    / "2_processed_data"
    / "6_combined_convergence"
    / "0_combined_final_sign_interactions.csv",
    index_col=0,
)
final_conv = final_conv[
    (final_conv["convergence"] > 2) & (final_conv["pvalue"] <= 0.05)
]

# Per genus, cluster the convergent TCRs and add cluster information to base dataframe
for i in final_conv["genus"].unique():
    print(i)
    dataframe = final_conv[final_conv["genus"] == i]
    get_clusters(
        dataframe, i, base_dir=base_dir / "2_processed_data" / "7_cluster_convergence"
    )

Desulfovibrio
Loading existing clustering results for Desulfovibrio...
Coprobacter
Loading existing clustering results for Coprobacter...
Coprococcus
Loading existing clustering results for Coprococcus...
Erysipelotrichaceae
Loading existing clustering results for Erysipelotrichaceae...
Faecalibacterium
Loading existing clustering results for Faecalibacterium...
Odoribacter
Loading existing clustering results for Odoribacter...
Prevotella
Loading existing clustering results for Prevotella...
Parasutterella
Loading existing clustering results for Parasutterella...
Ruminococcus
Loading existing clustering results for Ruminococcus...
Dorea
Loading existing clustering results for Dorea...
Phascolarctobacterium
Loading existing clustering results for Phascolarctobacterium...
Collinsella
Loading existing clustering results for Collinsella...
Paraprevotella
Loading existing clustering results for Paraprevotella...
Bifidobacterium
Loading existing clustering results for Bifidobacterium...
Sutt

In [3]:
# Sort the 36 genera based on importance
sorted_genera = [
    "Peptoniphilus",
    "Lactobacillus",
    "Desulfovibrio",
    "Porphyromonas",
    "Erysipelotrichaceae",
    "Actinomyces",
    "Dialister",
    "Phascolarctobacterium",
    "Oxalobacter",
    "Paraprevotella",
    "Akkermansia",
    "Parasutterella",
    "Coprococcus",
    "Odoribacter",
    "Coprobacter",
    "Bilophila",
    "Bacteroidales",
    "Sutterella",
    "Prevotella",
    "Streptococcus",
    "Ruminococcaceae",
    "Eubacterium",
    "Escherichia",
    "Clostridium",
    "Pseudoflavonifractor",
    "Anaerostipes",
    "Collinsella",
    "Bifidobacterium",
    "Ruminococcus",
    "Lachnospiraceae",
    "Roseburia",
    "Alistipes",
    "Dorea",
    "Faecalibacterium",
    "Oscillibacter",
    "Parabacteroides",
]

# Explore the clustered convergent TCRs and identify shared clusters among the three dataset
results = analyze_clusters(
    genera=sorted_genera,
    cluster_dir=base_dir / "2_processed_data" / "7_cluster_convergence",
    output_dir=base_dir / "2_processed_data" / "6_combined_convergence",
    convergence_threshold=2,
    pvalue_threshold=0.05,
    top_percentile=10,
)